import library

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from google.colab import drive

mount google drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


lokasi dataset

In [ ]:
excel_path = "/content/drive/MyDrive/Hackathon AI/cleaned_data/clean_dataset_final.xlsx"

load semua dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_path = "/content/drive/MyDrive"

excel_path = None

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".xlsx") and "clean_dataset" in file:
            excel_path = os.path.join(root, file)
            print("Ditemukan:", excel_path)

if excel_path is None:
    raise FileNotFoundError("File clean_dataset tidak ditemukan di Google Drive.")

Ditemukan: /content/drive/MyDrive/Hackathon AI /cleaned_data/clean_dataset.xlsx
Ditemukan: /content/drive/MyDrive/Hackathon AI /cleaned_data/clean_dataset_final.xlsx


load 13 datasett


In [ ]:
xls = pd.ExcelFile(excel_path)

cleaned_data = {}

for sheet in xls.sheet_names:
    cleaned_data[sheet] = pd.read_excel(xls, sheet_name=sheet)

print(f"Total Sheet: {len(cleaned_data)}")
print(list(cleaned_data.keys()))

Total Sheet: 13
['hotel-resto-v1', 'resto-hotel-v2', 'tempat-wisata-v1', 'wisata-v2', 'wisata-metadata', 'hotel-metadata', 'resto-metadata', 'transportasi', 'kuliner', 'Attractions Info', 'Artikel Danau Toba', 'Info Seputar Danau Toba (TOP 3)', 'waktu operasional destinasi']


document builer

In [ ]:
import pandas as pd

def row_to_document(sheet_name, row):
    lines = [f"Sumber: {sheet_name}"]

    for col, value in row.items():
        if pd.notna(value):
            value = str(value).strip()
            if value != "":
                lines.append(f"{col}: {value}")

    return "\n".join(lines)

generate knowlaedge base

In [ ]:
documents = []

for sheet_name, df in cleaned_data.items():

    print(f"Processing {sheet_name} ...")

    for idx, row in df.iterrows():

        documents.append({
            "id": f"{sheet_name}_{idx}",
            "sheet": sheet_name,
            "content": row_to_document(sheet_name, row)
        })

print(f"\nTotal Documents : {len(documents)}")

Processing hotel-resto-v1 ...
Processing resto-hotel-v2 ...
Processing tempat-wisata-v1 ...
Processing wisata-v2 ...
Processing wisata-metadata ...
Processing hotel-metadata ...
Processing resto-metadata ...
Processing transportasi ...
Processing kuliner ...
Processing Attractions Info ...
Processing Artikel Danau Toba ...
Processing Info Seputar Danau Toba (TOP 3) ...
Processing waktu operasional destinasi ...

Total Documents : 22760


preview

In [ ]:
print(documents[0]["content"])

Sumber: hotel-resto-v1
place: Labersa Hotel and Convention Center Toba, Balige
rating: 4,5
type: Hotel bintang 4
review: "Tempat nya bagus dan pemandangan yg bagus juga makan sangat baik"
htm: Mulai dari 850.091
facilitates: Wifi, Sarapan Gratis, Kolam renang luar, AC, Parkir Gratis, Layanan binatu
add: Saribu Raja Janji Maria, Kec. Balige, Toba, Sumatera Utara
service-hour: Waktu check out 12.00


simpan knowledge

In [ ]:
import json
import os

save_path = "/content/drive/MyDrive/Hackathon AI /knowledge_base"

os.makedirs(save_path, exist_ok=True)

with open(
    os.path.join(save_path, "knowledge_base.json"),
    "w",
    encoding="utf-8"
) as f:

    json.dump(documents, f, ensure_ascii=False, indent=2)

print("Knowledge Base berhasil disimpan.")

Knowledge Base berhasil disimpan.


In [ ]:
print(len(documents))

22760


In [ ]:
import os

file_path = "/content/drive/MyDrive/Hackathon AI /knowledge_base/knowledge_base.json"

print(os.path.getsize(file_path)/1024/1024, "MB")

7.0656280517578125 MB


In [ ]:
import json
import os

save_path = "/content/drive/MyDrive/Hackathon AI /knowledge_base"

os.makedirs(save_path, exist_ok=True)

with open(
    os.path.join(save_path, "knowledge_base.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print("Knowledge Base berhasil disimpan di:")
print(os.path.join(save_path, "knowledge_base.json"))

Knowledge Base berhasil disimpan di:
/content/drive/MyDrive/Hackathon AI /knowledge_base/knowledge_base.json


backfill

In [13]:
!pip install supabase -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.6 MB/s eta 0:00:00


In [14]:
from supabase import create_client

# Ganti dengan URL dan service_role key project Supabase kamu
# Ambil dari: Supabase Dashboard -> Settings -> API
SUPABASE_URL = "https://zzdqosceythaafmhhiga.supabase.co"
SUPABASE_SERVICE_KEY = "sb_secret_maJuMSrfh6mvew6LggGPqQ_dZX_Y4Pu"

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
print("Terkoneksi ke Supabase.")

Terkoneksi ke Supabase.


In [15]:
import re

TABLE_NAME = "documents"

# Sesuaikan lagi kalau ada nama sheet berbeda dari 13 sheet aslimu
CATEGORY_MAP = {
    "hotel-resto-v1": "hotel",
    "resto-hotel-v2": "resto",
    "tempat-wisata-v1": "wisata",
    "wisata-v2": "wisata",
    "wisata-metadata": "wisata",
    "hotel-metadata": "hotel",
    "resto-metadata": "resto",
    "transportasi": "transportasi",
    "kuliner": "kuliner",
    "Attractions Info": "wisata",
    "Artikel Danau Toba": "artikel",
    "Info Seputar Danau Toba (TOP 3)": "artikel",
    "waktu operasional destinasi": "jam_operasional",
}

RE_SUMBER = re.compile(r"^Sumber:\s*(.+)$", re.MULTILINE)
RE_RATING = re.compile(r"^rating:\s*(.+)$", re.MULTILINE)
RE_HTM = re.compile(r"^htm:\s*(.+)$", re.MULTILINE)
RE_HOUR = re.compile(r"^service-hour:\s*(.+)$", re.MULTILINE)
RE_ADD = re.compile(r"^add:\s*(.+)$", re.MULTILINE)
RE_PLACE = re.compile(r"^(?:place|place-name):\s*(.+)$", re.MULTILINE)


def parse_price(htm_str):
    if not htm_str:
        return None
    digits = re.sub(r"[^\d]", "", htm_str)
    return int(digits) if digits else None


def parse_rating(rating_str):
    if not rating_str:
        return None
    try:
        return float(rating_str.replace(",", "."))
    except ValueError:
        return None


def build_metadata(content: str) -> dict:
    sheet_match = RE_SUMBER.search(content)
    sheet = sheet_match.group(1).strip() if sheet_match else None

    rating_match = RE_RATING.search(content)
    htm_match = RE_HTM.search(content)
    hour_match = RE_HOUR.search(content)
    add_match = RE_ADD.search(content)
    place_match = RE_PLACE.search(content)

    metadata = {
        "sheet": sheet,
        "category": CATEGORY_MAP.get(sheet, "lainnya"),
    }

    if place_match:
        metadata["place"] = place_match.group(1).strip()
    if rating_match:
        r = parse_rating(rating_match.group(1).strip())
        if r is not None:
            metadata["rating"] = r
    if htm_match:
        p = parse_price(htm_match.group(1).strip())
        if p is not None:
            metadata["price_from"] = p
    if hour_match:
        metadata["service_hour"] = hour_match.group(1).strip()
    if add_match:
        metadata["address"] = add_match.group(1).strip()

    return metadata

print("Fungsi parsing metadata siap.")

Fungsi parsing metadata siap.


In [16]:
test_resp = supabase.table(TABLE_NAME).select("id, content").range(0, 2).execute()

for row in test_resp.data:
    metadata = build_metadata(row["content"] or "")
    print(f"id={row['id']}")
    print(metadata)
    print("---")
    supabase.table(TABLE_NAME).update({"metadata": metadata}).eq("id", row["id"]).execute()

print("Tes selesai. Cek tabel di Supabase Table Editor untuk konfirmasi.")

id=1
{'sheet': 'hotel-resto-v1', 'category': 'hotel', 'place': 'Labersa Hotel and Convention Center Toba, Balige', 'rating': 4.5, 'price_from': 850091, 'service_hour': 'Waktu check out 12.00', 'address': 'Saribu Raja Janji Maria, Kec. Balige, Toba, Sumatera Utara'}
---
id=2
{'sheet': 'hotel-resto-v1', 'category': 'hotel', 'place': 'Labersa Hotel and Convention Center Toba, Balige', 'rating': 4.5, 'price_from': 850091, 'service_hour': 'Waktu check out 12.00', 'address': 'Saribu Raja Janji Maria, Kec. Balige, Toba, Sumatera Utara'}
---
id=3
{'sheet': 'hotel-resto-v1', 'category': 'hotel', 'place': 'Damar Toba ~ Lakeside Eatery & Stay', 'rating': 4.8, 'price_from': 700000, 'service_hour': 'Waktu check in 14.00', 'address': 'Jl. Pemandian No.94, Kec. Balige, Sumatera Utara 22312'}
---
Tes selesai. Cek tabel di Supabase Table Editor untuk konfirmasi.


In [19]:
import os
import time

CHECKPOINT_FILE = "last_processed_id.txt"

BATCH_SIZE = 500

# ==========================================================
# LOAD LAST ID
# ==========================================================

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        last_id = int(f.read().strip())
else:
    last_id = 0

print(f"Resume from ID {last_id}")

total_updated = 0

# ==========================================================
# UPDATE METADATA
# ==========================================================

while True:

    response = (
        supabase
        .table(TABLE_NAME)
        .select("id, content")
        .gt("id", last_id)
        .order("id")
        .limit(BATCH_SIZE)
        .execute()
    )

    rows = response.data

    if not rows:
        break

    print(f"Fetched {len(rows)} rows...")

    for row in rows:

        try:

            metadata = build_metadata(row["content"] or "")

            (
                supabase
                .table(TABLE_NAME)
                .update({"metadata": metadata})
                .eq("id", row["id"])
                .execute()
            )

            last_id = row["id"]
            total_updated += 1

            # simpan checkpoint setiap berhasil update
            with open(CHECKPOINT_FILE, "w") as f:
                f.write(str(last_id))

        except Exception as e:

            print(f"Error pada ID {row['id']} : {e}")

            continue

    print(f"Last processed ID : {last_id}")

    time.sleep(0.1)

print(f"\nSelesai. Total updated : {total_updated}")

# hapus checkpoint jika seluruh proses selesai
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

Resume from ID 0
Fetched 500 rows...
Last processed ID : 500
Fetched 500 rows...
Last processed ID : 1000
Fetched 500 rows...
Last processed ID : 1500
Fetched 500 rows...
Last processed ID : 2000
Fetched 500 rows...
Last processed ID : 2500
Fetched 500 rows...
Last processed ID : 3000
Fetched 500 rows...
Last processed ID : 3500
Fetched 500 rows...
Last processed ID : 4000
Fetched 500 rows...
Last processed ID : 4500
Fetched 500 rows...
Last processed ID : 5000
Fetched 500 rows...
Last processed ID : 5500
Fetched 500 rows...
Last processed ID : 6000
Fetched 500 rows...
Last processed ID : 6500
Fetched 500 rows...
Last processed ID : 7000
Fetched 500 rows...
Last processed ID : 7500
Fetched 500 rows...
Last processed ID : 8000
Fetched 500 rows...
Last processed ID : 8500
Fetched 500 rows...
Last processed ID : 9000
Fetched 500 rows...
Last processed ID : 9500
Fetched 500 rows...
Error pada ID 9980 : <ConnectionTerminated error_code:0, last_stream_id:19999, additional_data:None>
Last pro